<a href="https://colab.research.google.com/github/kristen531/Fall-Detection-Models/blob/main/prediction1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [123]:
!pip install torch torch_geometric pandas --quiet


In [124]:
from google.colab import files
uploaded = files.upload()

Saving edges.csv to edges (4).csv
Saving nodes.csv to nodes (5).csv


In [144]:
import pandas as pd
import torch

# 读取 CSV 文件
nodes_df = pd.read_csv("nodes.csv")
edges_df = pd.read_csv("edges.csv")

# 去掉字段前缀 "p."
nodes_df.columns = nodes_df.columns.str.replace("p.", "", regex=False)

# 显示确认
print("📍 节点前几行:")
print(nodes_df.head())
print("\n📍 边前几行:")
print(edges_df.head())



📍 节点前几行:
   node_id patient_id health_risk  gender       created_at
0       46      P1000         low    Male   1/19/2023 8:00
1       47      P1001      medium  Female   1/19/2023 8:30
2       48      P1002      medium  Female  1/19/2023 10:00
3       49      P1003         low    Male   1/20/2023 8:30
4       50      P1004         low  Female  1/20/2023 11:30

📍 边前几行:
   source  target                relation
0       0     146  USED_IN_RECOMMENDATION
1       1     147  USED_IN_RECOMMENDATION
2       2     148  USED_IN_RECOMMENDATION
3       3     149  USED_IN_RECOMMENDATION
4       4     150  USED_IN_RECOMMENDATION


In [145]:
# 删除 health_risk 是空的行（否则会变成 -1 标签）
nodes_df = nodes_df.dropna(subset=["health_risk"]).reset_index(drop=True)

# 重新编号节点：mapped_node_id 从 0 ~ N-1
nodes_df["mapped_node_id"] = range(len(nodes_df))

# 建立原始 node_id → 新 ID 的映射
id_mapping = dict(zip(nodes_df["node_id"], nodes_df["mapped_node_id"]))

# 替换边的 source 和 target 成新 ID
edges_df["source"] = edges_df["source"].map(id_mapping)
edges_df["target"] = edges_df["target"].map(id_mapping)

# 移除无效边（有 NaN）
edges_df = edges_df.dropna().astype(int).reset_index(drop=True)



In [146]:
from torch_geometric.data import Data

# 特征 (gender → one-hot)
X = pd.get_dummies(nodes_df["gender"])
x = torch.tensor(X.values, dtype=torch.float)

# 标签 (health_risk → 分类编号)
y = nodes_df["health_risk"].astype("category").cat.codes
y = torch.tensor(y.values, dtype=torch.long)

# 边索引
edge_index = torch.tensor(edges_df[["source", "target"]].values.T, dtype=torch.long)

# 构建图数据
data = Data(x=x, edge_index=edge_index, y=y)

print("\n✅ 图数据载入成功:")
print(data)



✅ 图数据载入成功:
Data(x=[26, 2], edge_index=[2, 0], y=[26])


In [147]:
import torch.nn as nn
from torch_geometric.nn import GATConv

class GATModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GATModel, self).__init__()
        self.gat1 = GATConv(input_dim, hidden_dim, heads=4, dropout=0.6)
        self.gat2 = GATConv(hidden_dim * 4, output_dim, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = self.gat1(x, edge_index)
        x = torch.relu(x)
        x = self.gat2(x, edge_index)
        return x


In [148]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GATModel(input_dim=data.num_node_features, hidden_dim=8, output_dim=len(torch.unique(data.y))).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()


In [149]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out, data.y)
    loss.backward()
    optimizer.step()
    return loss.item()

# 训练 200 次
for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}")


Epoch 020, Loss: 1.0714
Epoch 040, Loss: 1.0858
Epoch 060, Loss: 1.0727
Epoch 080, Loss: 1.0817
Epoch 100, Loss: 1.0951
Epoch 120, Loss: 1.0736
Epoch 140, Loss: 1.0927
Epoch 160, Loss: 1.0796
Epoch 180, Loss: 1.1005
Epoch 200, Loss: 1.0957


In [150]:
model.eval()
out = model(data.x, data.edge_index)
preds = out.argmax(dim=1)

# 显示前 10 个预测结果
print("📍 前10个预测标签：", preds[:10].tolist())


📍 前10个预测标签： [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [151]:
!pip install neo4j --quiet


In [157]:
from neo4j import GraphDatabase

# ⚠️ 请替换成你自己的 Neo4j Aura 信息
uri = "neo4j+s://286a1468.databases.neo4j.io"
username = "neo4j"
password = "JoZo-BELo0w7KEg9lVseaXMqx4XfEQx-wfnXu2soMpM"

driver = GraphDatabase.driver(uri, auth=(username, password))


# 测试连接
try:
    driver = GraphDatabase.driver(uri, auth=(username, password))
    with driver.session() as session:
        result = session.run("RETURN '✅ Neo4j 连接成功！' AS message")
        print(result.single()["message"])
except Exception as e:
    print("❌ 连接失败，请检查 URI、账号或密码")
    print(e)


✅ Neo4j 连接成功！


In [158]:
# 将 mapped_node_id → 原始 node_id 的映射找回
mapped_to_original = dict(zip(nodes_df["mapped_node_id"], nodes_df["node_id"]))

# 写入每个节点的预测值
with driver.session() as session:
    for i, pred in enumerate(preds.tolist()):
        original_id = int(mapped_to_original[i])
        session.run("""
            MATCH (n) WHERE id(n) = $id
            SET n.prediction = $pred
        """, id=original_id, pred=int(pred))

print("✅ 预测结果已成功写入 Neo4j！")


✅ 预测结果已成功写入 Neo4j！
